In [1]:
!python -m pip install pyserini

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.3/413.3 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.4/197.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0

In [3]:
!apt-get update
!apt-get install -y openjdk-21-jdk-headless

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

In [4]:
import os
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from pyserini.search import get_topics

# 1. Setup
output_dir = "results/baseline"
os.makedirs(output_dir, exist_ok=True)
run_file = f"{output_dir}/run.miracl.bm25.ar.txt"

# 2. Load Resources
print("Loading Index and Topics...")
# We re-declare this to ensure it's fresh if you restarted
searcher = LuceneSearcher.from_prebuilt_index('miracl-v1.0-ar')
searcher.set_language('arabic')

topics = get_topics('miracl-v1.0-ar-dev')
# Topics format: {qid: {'title': 'query text'}}

# 3. Execution Loop
print(f"Running BM25 on {len(topics)} queries...")

with open(run_file, 'w') as f:
    # tqdm makes a progress bar so you know it's working
    for qid in tqdm(topics.keys(), desc="Retrieving"):
        query_text = topics[qid]['title']

        # Retrieve top 10 (We focus on Recall@10)
        hits = searcher.search(query_text, k=100)

        for i, hit in enumerate(hits):
            # TREC Format: qid Q0 docid rank score run_id
            # Standard format required for evaluation tools
            f.write(f"{qid} Q0 {hit.docid} {i+1} {hit.score:.5f} baseline_bm25\n")

print(f"\n✅ Retrieval complete. Results saved to: {run_file}")

Loading Index and Topics...
Running BM25 on 2896 queries...


Retrieving: 100%|██████████| 2896/2896 [00:22<00:00, 125.92it/s]


✅ Retrieval complete. Results saved to: results/baseline/run.miracl.bm25.ar.txt


In [5]:
!pip install pytrec_eval

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309346 sha256=2cb5fb135304d1cc711f427416adf281e19a7f793138a4b79d9f76c17b6658bf
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [6]:
import pytrec_eval
import json
from pyserini.search import get_qrels

# 1. Load the Ground Truth (Qrels)
raw_qrels = get_qrels('miracl-v1.0-ar-dev')

# --- FIX: Convert all keys to Strings ---
# pytrec_eval crashes if Query IDs or Doc IDs are not strings
qrels = {}
for qid, doc_dict in raw_qrels.items():
    # Convert Query ID to string
    str_qid = str(qid)
    qrels[str_qid] = {}

    for docid, score in doc_dict.items():
        # Ensure Doc ID is string and Score is int
        qrels[str_qid][str(docid)] = int(score)
# ----------------------------------------

# 2. Load Your Run File
# We must also ensure the run_data uses String IDs
with open(f"{output_dir}/run.miracl.bm25.ar.txt", 'r') as f:
    run_data = pytrec_eval.parse_run(f)

# Ensure run_data keys match qrels keys (Strings)
# Sometimes parse_run infers types, so let's be safe:
run_data = {str(k): v for k, v in run_data.items()}

# 3. Define Metrics
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels, {'recall_100', 'ndcg_cut_10', 'recip_rank'}
)

# 4. Compute Scores
results = evaluator.evaluate(run_data)

# 5. Aggregate Results
def print_aggregate(results):
    metrics = ['recall_100', 'ndcg_cut_10', 'recip_rank']
    aggregates = {m: 0.0 for m in metrics}

    count = 0
    for qid in results:
        count += 1
        for m in metrics:
            aggregates[m] += results[qid][m]

    # Calculate average
    if count > 0:
        for m in metrics:
            aggregates[m] /= count

    print("-" * 30)
    print("BASELINE PERFORMANCE (BM25)")
    print("-" * 30)
    print(f"Recall@100: {aggregates['recall_100']:.4f}")
    print(f"NDCG@10:   {aggregates['ndcg_cut_10']:.4f}")
    print(f"MRR:       {aggregates['recip_rank']:.4f}")
    print("-" * 30)

print_aggregate(results)

------------------------------
BASELINE PERFORMANCE (BM25)
------------------------------
Recall@100: 0.2700
NDCG@10:   0.1057
MRR:       0.1090
------------------------------
